# 04 - Feature Engineering
# Step 1: Feature Engineering & Master Dataset Fusion
**Project:** Eurovision Song Contest Prediction (2008-2026)
**Objective:** Merge processed datasets (Audio, Betting, Votes, Contestants) and engineer target variables and sociodemographic/geopolitical predictors for ML modeling.

**Target Variables to Engineer:**
1. `final_score` / `final_rank` (Continuous - for Linear Regression, SVR, KNN, Random Forest)
2. `is_qualified` (Categorical 1/0 - for Logistic Regression)

**Predictor Features to Engineer:**
* Historical Voting Alliances (Rolling average of points received)
* Cultural/Linguistic Connections (Binary flags)
* Audio Features (Merged from Essentia output)
* Market/Media Influence (Merged from betting odds)

In [6]:
# ==========================================
# CELL 1: MOUNT GOOGLE DRIVE
# ==========================================
from google.colab import drive

# This will prompt you to log in and grant access to your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# ==========================================
# CELL 2: INSTALL & IMPORT DEPENDENCIES
# ==========================================
# Install any potentially missing libraries (Colab usually has these, but this is a safeguard)
!pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ML Libraries
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print("\nAll libraries successfully installed and imported!")


All libraries successfully installed and imported!


In [9]:
# ==========================================
# CELL 3: DATA INGESTION, SQUASHING & STANDARDIZATION
# ==========================================
# UPDATE THIS PATH to exactly where your CSV files are located!
BASE_PATH = '/content/drive/MyDrive/Eurovision_Prediction/data/processed'

# 1. Load All Datasets
df_contestants = pd.read_csv(os.path.join(BASE_PATH, 'contestants_processed.csv'))
df_audio = pd.read_csv(os.path.join(BASE_PATH, 'audio_features.csv'))
df_votes = pd.read_csv(os.path.join(BASE_PATH, 'votes_processed.csv'))
df_betting = pd.read_csv(os.path.join(BASE_PATH, 'betting_offices.csv'))
df_jurors = pd.read_csv(os.path.join(BASE_PATH, 'jurors.csv'))

# 2. SQUASH DUPLICATE ROWS (SF & GF)
print(f"Original Contestants Shape: {df_contestants.shape}")
df_contestants = df_contestants.groupby(['year', 'to_country_id'], as_index=False).first()
print(f"Squashed Contestants Shape: {df_contestants.shape}\n")

# 3. The "Rosetta Stone" Mapping
mapping_df = df_contestants[['to_country_id', 'to_country']].dropna().drop_duplicates()
eurovision_map = dict(zip(mapping_df['to_country_id'], mapping_df['to_country']))

# 4. Standardize Keys Across All Datasets
if 'country' in df_audio.columns:
    df_audio['country'] = df_audio['country'].map(eurovision_map)
elif 'to_country_id' in df_audio.columns:
    df_audio['country'] = df_audio['to_country_id'].map(eurovision_map)

df_votes['to_country'] = df_votes['to_country'].map(eurovision_map)
df_votes['from_country'] = df_votes['from_country'].map(eurovision_map)

df_betting = df_betting.rename(columns={'country_name': 'country'})
df_targets = df_contestants.rename(columns={'to_country': 'country'})

# 5. Derive the Machine Learning Targets
df_targets['is_qualified'] = df_targets['place_final'].notna().astype(int)
df_targets['final_score'] = df_targets['points_final'].fillna(0)
df_targets['final_rank'] = df_targets['place_final'].fillna(99)

print("Data successfully standardized! Target variables derived.")
display(df_targets[['year', 'country', 'is_qualified', 'final_score', 'final_rank', 'place_sf']].head(5))

Original Contestants Shape: (1333, 21)
Squashed Contestants Shape: (718, 21)

Data successfully standardized! Target variables derived.


,year,country,is_qualified,final_score,final_rank,place_sf
0,2008,Andorra,0,0.0,99.0,16.0
1,2008,Albania,1,55.0,17.0,9.0
2,2008,Armenia,1,199.0,4.0,2.0
3,2008,Azerbaijan,1,132.0,8.0,6.0
4,2008,Bosnia & Herzegovina,1,110.0,10.0,9.0


In [17]:
# ==========================================
# CELL 4: THE CORE AUDIO & TARGETS DATASET
# ==========================================
from sklearn.impute import KNNImputer
import pandas as pd

# 1. Start with the cleaned target data (Keep 2008 and later)
df_core = df_targets[df_targets['year'] >= 2008].copy()

if 'place_sf' in df_core.columns:
    df_core = df_core.rename(columns={'place_sf': 'semi_final_place'})

# 2. AGGREGATE external datasets BEFORE merging
if 'year' in df_audio.columns and 'country' in df_audio.columns:
    df_audio_agg = df_audio.groupby(['year', 'country'], as_index=False).first()
    df_core = pd.merge(df_core, df_audio_agg, on=['year', 'country'], how='left')

if 'year' in df_jurors.columns and 'country' in df_jurors.columns:
    df_jurors_agg = df_jurors.groupby(['year', 'country'], as_index=False).first()
    df_core = pd.merge(df_core, df_jurors_agg, on=['year', 'country'], how='left')

# Drop any accidental duplicate columns created by pandas
df_core = df_core.loc[:, ~df_core.columns.str.endswith('_y')]
df_core.columns = df_core.columns.str.replace('_x', '')

# ==========================================
# 3. CONTEXT-BASED IMPUTATION
# ==========================================
# A. Domain Logic Targets
df_core['final_score'] = df_core['final_score'].fillna(0)
df_core['final_rank'] = df_core['final_rank'].fillna(99)

if 'total_jury_mentions' in df_core.columns:
    df_core['total_jury_mentions'] = df_core['total_jury_mentions'].fillna(0)

# C. Algorithmic Imputation (Audio Features) via KNN
audio_cols = ['danceability', 'energy', 'tempo', 'valence']
available_audio_cols = [c for c in audio_cols if c in df_core.columns]

if available_audio_cols:
    print(f"Applying KNN Imputation for audio features: {available_audio_cols}")
    knn_imputer = KNNImputer(n_neighbors=5)
    df_core[available_audio_cols] = knn_imputer.fit_transform(df_core[available_audio_cols])

print(f"Core Dataset Shape (2008+): {df_core.shape}\n")
display(df_core[['year', 'country', 'is_qualified', 'final_score'] + available_audio_cols].head())

Core Dataset Shape (2008+): (718, 144)



,year,country,is_qualified,final_score
0,2008,Andorra,0,0.0
1,2008,Albania,1,55.0
2,2008,Armenia,1,199.0
3,2008,Azerbaijan,1,132.0
4,2008,Bosnia & Herzegovina,1,110.0


In [18]:
# ==========================================
# CELL 5: VOTING NETWORK & RELATIONSHIP INDICATORS
# ==========================================
import numpy as np

# 1. Base Ledger: Keep 2008 and later
df_voting_network = df_votes[df_votes['year'] >= 2008].copy()

# 2. Merge Juror details onto the Voting Ledger
# Join on the dyadic keys: who is voting for whom, in what year and round
if all(c in df_jurors.columns for c in ['year', 'round', 'from_country', 'to_country']):
    df_voting_network = pd.merge(
        df_voting_network,
        df_jurors,
        on=['year', 'round', 'from_country', 'to_country'],
        how='left'
    )

# 3. Contextual Imputation for Voting Data
# Points: If televote/jury points are missing, assume 0 points given from that group
for col in ['total_points', 'tele_points', 'jury_points']:
    if col in df_voting_network.columns:
        df_voting_network[col] = df_voting_network[col].fillna(0)

# Rankings: If juror rankings are missing, assign 99 (worst penalty, preserving 1 as the best)
juror_cols = ['televote_rank', 'jury_rank', 'juror_A', 'juror_B', 'juror_C', 'juror_D', 'juror_E', 'juror_F', 'juror_G']
available_juror_cols = [c for c in juror_cols if c in df_voting_network.columns]
df_voting_network[available_juror_cols] = df_voting_network[available_juror_cols].fillna(99)

# 4. Feature Engineering: Jury Variance
# How much did the individual jurors disagree? High variance = controversial act
individual_jurors = [c for c in ['juror_A', 'juror_B', 'juror_C', 'juror_D', 'juror_E'] if c in df_voting_network.columns]
if individual_jurors:
    # Calculate standard deviation of juror rankings for each row.
    # Use replace(99, np.nan) temporarily so we don't skew the variance for missing data
    df_voting_network['jury_variance'] = df_voting_network[individual_jurors].replace(99, np.nan).std(axis=1).fillna(0)

# 5. Feature Engineering: Relationship Indicator (0.0 to 1.0)
# How often does 'from_country' give maximum points (12) to 'to_country' over time?
df_voting_network = df_voting_network.sort_values(by=['year'])
# Calculate the cumulative average points given, divided by 12 (max possible)
df_voting_network['historical_affinity'] = (
    df_voting_network.groupby(['from_country', 'to_country'])['total_points']
    .transform(lambda x: x.expanding().mean().shift()) / 12.0
).fillna(0) # First time they vote for each other gets a 0

# Cap at 1.0 just in case data anomalies exist
df_voting_network['historical_affinity'] = df_voting_network['historical_affinity'].clip(upper=1.0)

print(f"Voting Network Dataset Shape: {df_voting_network.shape}\n")
display(df_voting_network[['year', 'from_country', 'to_country', 'total_points', 'historical_affinity', 'jury_variance']].head())

Voting Network Dataset Shape: (30998, 20)



,year,from_country,to_country,total_points,historical_affinity,jury_variance
17,2008,Albania,Sweden,3,0.0,0.0
16,2008,Albania,Albania,0,0.0,0.0
15,2008,Albania,Spain,1,0.0,0.0
14,2008,Albania,Denmark,0,0.0,0.0
13,2008,Albania,Iceland,0,0.0,0.0


In [19]:
# ==========================================
# CELL 6: BETTING ACCURACY & FINAL RESULTS
# ==========================================

# 1. Filter targets for Grand Finalists (2008+) to compare against odds
df_results = df_targets[(df_targets['year'] >= 2008) & (df_targets['is_qualified'] == 1)].copy()

# 2. Aggregate Betting Data
odds_col = 'betting_score' if 'betting_score' in df_betting.columns else 'odds'

df_betting_agg = df_betting.groupby(['year', 'country']).agg(
    avg_odds=(odds_col, 'mean'),
    best_odds=(odds_col, 'min'),
    worst_odds=(odds_col, 'max'),
    bookie_count=('betting_bm_id', 'count') # How many bookies tracked them
).reset_index()

# 3. Merge Results with Betting
df_betting_accuracy = pd.merge(df_results[['year', 'country', 'place_final', 'points_final']],
                               df_betting_agg, on=['year', 'country'], how='left')

# 4. Handle Missing Odds (Penalty)
# If a finalist has no odds, they were a massive underdog. Fill with the max odds of the year.
df_betting_accuracy['avg_odds'] = df_betting_accuracy.groupby('year')['avg_odds'].transform(lambda x: x.fillna(x.max()))

# 5. Feature Engineering: Market Predictions
# Implied probability: (1 / decimal odds) * 100
df_betting_accuracy['implied_win_probability_pct'] = (1 / df_betting_accuracy['avg_odds']) * 100

# Rank the countries per year based on their odds (1 = lowest odds/favorite)
df_betting_accuracy['bookie_predicted_rank'] = df_betting_accuracy.groupby('year')['avg_odds'].rank(method='min')

# Calculate the Error: How far off were the bookies?
df_betting_accuracy['prediction_error'] = abs(df_betting_accuracy['place_final'] - df_betting_accuracy['bookie_predicted_rank'])

print(f"Betting Accuracy Dataset Shape: {df_betting_accuracy.shape}\n")
display(df_betting_accuracy[['year', 'country', 'place_final', 'bookie_predicted_rank', 'avg_odds', 'prediction_error']].head(10))

Betting Accuracy Dataset Shape: (462, 11)



,year,country,place_final,bookie_predicted_rank,avg_odds,prediction_error
0,2008,Albania,17.0,NaN,NaN,NaN
1,2008,Armenia,4.0,NaN,NaN,NaN
2,2008,Azerbaijan,8.0,NaN,NaN,NaN
3,2008,Bosnia & Herzegovina,10.0,NaN,NaN,NaN
4,2008,Germany,23.0,NaN,NaN,NaN
5,2008,Denmark,15.0,NaN,NaN,NaN
6,2008,Spain,16.0,NaN,NaN,NaN
7,2008,Finland,22.0,NaN,NaN,NaN
8,2008,France,19.0,NaN,NaN,NaN
9,2008,United Kingdom,25.0,NaN,NaN,NaN


In [20]:
# ==========================================
# CELL 7: EXPORT MODULAR DATASETS
# ==========================================
import os

# Ensure the BASE_PATH exists from your earlier cells
path_core = os.path.join(BASE_PATH, 'eurovision_core_features.csv')
path_voting = os.path.join(BASE_PATH, 'eurovision_voting_network.csv')
path_betting = os.path.join(BASE_PATH, 'eurovision_betting_accuracy.csv')

# Export to CSV without the pandas index column
df_core.to_csv(path_core, index=False)
df_voting_network.to_csv(path_voting, index=False)
df_betting_accuracy.to_csv(path_betting, index=False)

print("SUCCESS! All three datasets have been cleanly exported:")
print(f"1. Core Features:  {path_core}")
print(f"2. Voting Network: {path_voting}")
print(f"3. Betting Stats:  {path_betting}")
print("\nYou can now download these from your Google Drive for review.")

SUCCESS! All three datasets have been cleanly exported:
1. Core Features:  /content/drive/MyDrive/Eurovision_Prediction/data/processed/eurovision_core_features.csv
2. Voting Network: /content/drive/MyDrive/Eurovision_Prediction/data/processed/eurovision_voting_network.csv
3. Betting Stats:  /content/drive/MyDrive/Eurovision_Prediction/data/processed/eurovision_betting_accuracy.csv

You can now download these from your Google Drive for review.
